## 13. 예상 질문 정리

Q. 왜 처음부터 학습 안 하고 pretrained를 썼나?
train이 4만 장(클래스당 400장)이라 처음부터 사물 보는 법을 배우기엔 데이터가 부족함. ImageNet으로 미리 학습된 모델은 선, 모양, 질감 같은 특징을 이미 알고 있어서 마지막 층만 바꾸고 조금 더 학습하면(fine-tuning) 적은 데이터로도 잘 나옴. 이게 transfer learning.

Q. pretrained 쓰는 게 규칙 위반 아닌가?
Q&A(A1, A5, A10)에서 외부 데이터셋으로 사전학습된 공개 weight를 가져와서 주어진 train set으로 fine-tuning하는 건 설명만 할 수 있으면 된다고 하셨음. 대신 11/21 추가된 조건(A1, A5, A8, A12)으로 온라인 원본 CIFAR-100 데이터랑 CIFAR-100으로 학습된 모델은 못 씀. 여기선 ImageNet(22K -> 1K), LVD-142M(DINOv2)으로 학습된 weight만 썼고, 데이터도 제공된 train/test 파일만 씀. 100개 클래스 분류층은 둘 다 이 session에서 새로 학습함.

Q. DINOv2 논문에 CIFAR-100 결과가 있던데 CIFAR-100으로 학습된 모델 아닌가?
논문의 CIFAR-100 수치는 backbone을 고정해두고 그 위에 분류층을 따로 붙여서 잰 평가 결과임. 그 분류층은 공개 weight에 안 들어있음. 쓴 건 LVD-142M으로 label 없이 학습한 backbone뿐이고(num_classes=0으로 분류층 없이 불러옴), MLP head는 제공된 train set으로 이 session에서 처음부터 학습함.

Q. 왜 1K 말고 22K로 학습된 weight를 썼나?
구조는 같은데 이미지를 훨씬 많이(1400만 장) 보고 학습한 weight라 일반적인 특징을 더 잘 앎. 1K weight(v1)랑 같은 설정으로 돌려서 비교해보고 더 좋은 쪽 고름.

Q. 왜 ConvNeXt-Tiny?
ResNet에 요즘 기법(큰 depthwise conv, LayerNorm, GELU 등)을 넣은 CNN이라 ResNet-50보다 정확한데 크기(28M 정도)는 비슷함. attention 쓰는 모델보다 가볍고, CNN이라 수업에서 배운 conv, pooling, skip connection으로 설명 가능함.

Q. 왜 32x32를 224x224로 키우나?
pretrained 모델이 224로 학습돼서. 32를 그대로 넣으면 첫 층(stride 4)에서 8x8 되고 마지막엔 1x1보다 작아져서 공간 정보가 없어짐. 키워서 넣으면 모델이 익숙한 크기로 보임.

Q. 정규화 값을 왜 ImageNet 걸로 바꿨나?
pretrained 모델이 ImageNet 평균/표준편차로 정규화된 입력으로 학습돼서, 입력 분포를 똑같이 맞춰줘야 첫 층부터 학습한 대로 동작함.

Q. depthwise conv랑 1x1 conv가 뭔가?
depthwise conv는 채널마다 따로 kernel 걸어서 공간 정보만 보고, 1x1 conv는 위치마다 채널끼리 섞어서 채널 정보만 봄. 일반 conv가 한 번에 하던 걸 둘로 나눈 거라 계산량이 적고, 그래서 7x7처럼 큰 kernel도 쓸 수 있음.

Q. LayerNorm이랑 BatchNorm 차이?
BatchNorm은 batch 안 여러 이미지에 걸쳐 채널별 평균/분산을 구하고, LayerNorm은 이미지 한 장 안에서 (위치마다) 채널 방향으로 구함. LayerNorm은 batch 크기 영향이 없고 train/eval 동작도 같음.

Q. lr을 왜 1e-4로 작게? warmup은 왜?
pretrained weight는 이미 좋은 상태라 lr이 크면 배운 특징이 망가짐. 그리고 새로 바꾼 분류층이 랜덤이라 초반 gradient가 커서 처음 10% 동안 lr을 천천히 올림(warmup).

Q. AdamW랑 Adam 차이?
Adam은 weight decay를 gradient에 더해서 적용하는데, 그러면 parameter마다 decay 세기가 달라져버림(적응적 lr 때문에). AdamW는 weight decay를 따로 빼서 가중치에서 직접 일정 비율을 빼니까 의도한 대로 규제가 됨.

Q. TrivialAugment가 뭔가?
회전, 밝기, 대비, 기울이기 같은 변환 중에 매번 하나를 랜덤으로 고르고 세기도 랜덤으로 정해서 적용함. 정할 값이 없고 다양하게 변형돼서 overfitting 줄여줌.

Q. label smoothing은 왜?
정답을 1.0으로 맞추게 학습하면 train 이미지에 너무 확신하게 됨. 정답 0.9, 나머지 0.1로 나눈 target으로 학습해서 덜 확신하게 하면 처음 보는 이미지에서 더 잘 맞춤.

Q. AMP랑 GradScaler가 뭔가?
AMP는 계산 일부를 16bit로 해서 T4에서 빠르게 돌리고 메모리 아끼는 것. 16bit는 범위가 좁아서 작은 gradient가 0이 될 수 있는데, GradScaler가 loss에 큰 수를 곱해서 backward하고 업데이트 전에 다시 나눠줌.

Q. stochastic depth가 뭔가?
학습 중에 블록을 확률적으로 건너뛰어서(skip connection으로 x만 통과) 매번 조금씩 다른 깊이로 학습함. 특정 블록에 의존 안 하게 하는 규제고, 평가 때는 블록 다 씀. 블록 단위 Dropout이랑 비슷함.

Q. validation은 왜 나눴나?
test 정답이 없고 public score는 하루 10번만 볼 수 있어서. train 일부로 직접 성능 보고 제일 좋은 epoch를 고름. public score(test의 30%)에만 맞추게 되는 것도 피할 수 있음.

Q. TTA가 뭔가?
추론할 때 원본이랑 좌우 반전을 각각 예측해서 점수 더하는 것. 한쪽에서 헷갈린 걸 다른 쪽이 메꿔줘서 예측이 안정적임.

Q. ensemble은 뭐고, 왜 softmax 확률로 평균 내나?
여러 모델 예측을 평균 내서 최종 예측 정하는 것. 모델마다 틀리는 이미지가 달라서 서로 실수를 메꿔줌. logits는 모델마다 크기가 달라서 그냥 더하면 한 모델이 다 정해버림. softmax로 0~1 확률로 맞추고 평균 내야 같은 비중으로 들어감.

Q. model.train()이랑 model.eval() 차이?
이 모델에선 stochastic depth가 다르게 동작함. train에선 블록을 확률적으로 건너뛰고 eval에선 다 씀. 평가 때 eval로 안 바꾸면 결과가 매번 달라짐.

Q. DINOv2가 뭔가?
label 없이 이미지만으로 학습한(self-supervised) ViT. 같은 이미지에서 자른 다른 crop을 넣었을 때 비슷한 표현이 나오게 학습해서 특정 데이터셋 label에 안 치우친 feature가 나옴. 1억 4천만 장으로 학습돼서 다른 데이터셋에서 Linear 하나만 얹어도 정확도가 높음.

Q. DINOv2는 왜 fine-tuning 안 하고 frozen으로?
ViT-g는 parameter가 11억 개라 T4에서 fine-tuning하면 메모리랑 session 시간 안에 안 끝남. frozen이면 forward 한 번만 하면 되고 head 학습은 몇 분이면 됨. 실험에서도 이 방식(Linear head val 93.75%)이 ConvNeXt 전체 fine-tuning(91.03%)보다 높았음.

Q. CLS token이랑 patch token이 뭔가?
ViT는 이미지를 14x14 patch로 잘라서 각각 vector(patch token)로 만들고, 맨 앞에 전체 요약용 CLS token을 붙임. self-attention 거치면서 CLS에는 전체 정보가 모이고 patch token엔 각 위치 정보가 남음. 둘 다 쓰면 요약 정보랑 골고루 퍼진 정보를 같이 쓸 수 있음.

Q. 왜 마지막 4개 블록 CLS token을 이어 붙였나?
블록마다 이미지 표현이 조금씩 달라서(뒤로 갈수록 추상적) 같이 쓰면 head가 쓸 정보가 늘어남. DINOv2 논문 linear 평가 방식 따라한 거고, 실험에서 마지막 블록만 쓴 것(93.67%)보다 조금 높았음(93.75%). 같은 forward에서 꺼내서 시간은 안 늘어남.

Q. 왜 Linear 말고 MLP head?
Linear는 feature 공간을 평면으로만 나눔. hidden layer + GELU 하나 넣으면 휘어진 경계가 돼서 Linear로 안 나눠지는 클래스도 나눌 수 있음. parameter가 늘어서 overfitting 걱정돼서 Dropout 0.5, weight decay 0.05 넣음. seed 3번 바꿔봐도 Linear보다 0.4~0.55%p 계속 높았음.

Q. pseudo-labeling은 왜 안 썼나?
확신 높은 test 예측을 정답처럼 넣고 다시 학습해봤는데 val이 안 올랐음. 같은 feature 쓰는 모델 자기 예측이라 새로 배울 게 거의 없고, 틀린 label(2~4%)이 섞이는 손해가 있었던 것 같음.

Q. forward hook이 뭔가?
특정 층(여기선 블록)이 forward 끝날 때마다 자동으로 불리는 함수. 모델 코드 안 고치고 중간 블록 출력을 꺼내려고 씀. 추출 끝나면 지움.

Q. DINOv2는 왜 augmentation 안 했나?
feature를 한 번 뽑아서 재사용하는 구조라 랜덤 augmentation 쓰려면 epoch마다 다시 뽑아야 함(한 번에 70분). 대신 결과가 항상 같은 좌우 반전 feature를 뽑아서 train을 2배로 늘림. 학습하는 건 작은 MLP head뿐이고 Dropout, weight decay도 있어서 overfitting 걱정은 적음.

Q. feature를 왜 표준화하나?
차원마다 범위가 다르면 큰 차원이 학습을 좌우하고 학습도 불안정해짐. train feature 평균/표준편차로 전부 평균 0, 표준편차 1로 맞추고 val, test에도 같은 값 씀.

Q. 왜 ConvNeXt랑 DINOv2를 ensemble했나?
CNN fine-tuning 모델이랑 ViT feature에 MLP head 얹은 모델은 보는 방식이 달라서 틀리는 이미지가 다름. 실험에서 ensemble이 94.77%로 각각(ConvNeXt 91.03%, DINOv2 MLP 94.30%)보다 높았음. 같은 ConvNeXt를 weight만 바꿔 섞었을 땐 거의 안 올랐음.

Q. 왜 전부 한 session 안에서 하나?
학습한 모델을 저장했다가 다른 session에서 불러 쓰는 건 자원 공정성 때문에 안 됨. 그래서 두 모델 학습, feature 추출, ensemble을 전부 노트북 한 번 실행 안에서 함. 같은 session 안에서 best weight(convnext_best.pt) 저장했다가 불러오는 건 괜찮음.